# S07 · How a model learns — the learning rate

So far `scikit-learn` found the best line instantly, in one `.fit()`. But what does *"the model learns"* actually mean? Here we open the box and watch the same line being found gradually — nudged downhill, step by step — until it settles on the answer the library gives. The size of each nudge is the **learning rate**, the single most important dial in modern machine learning. This downhill walk, **gradient descent**, is the engine behind almost every model in this course, right up to neural networks.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GirishMKulkarni/applied-maths-in-industry-site/blob/main/sessions/S07/notebooks/03_learning_step_by_step.ipynb)

*New here? Press each cell's play button, top to bottom, and read the plain-English note above it. Nothing to install on Colab.*

**New here? Read this once.**

- New to Python? You can still do this whole notebook. Press the play button on each cell, top to bottom, and read the plain-English note above it.
- Want the idea behind today in one page? Open `primers/least_squares_and_lines.md`.
- Already confident with code? Skip to the last **Your turn** cell.
- Stuck on a word? It is in `primers/glossary.md`.

## Setup

On **Google Colab**, run the next cell once. On your **own machine** you already installed everything with `uv`, so it does nothing there.

In [ ]:
# This notebook only uses numpy, pandas, matplotlib and scikit-learn,
# all of which Google Colab already ships, so there is nothing to install.
print("Setup complete - nothing to install.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression   # to check our answer at the end

## Step 1 — the same real data, one feature

We reuse the California housing data, predicting median house value from median income, so at the end we can compare our hand-cranked answer with `scikit-learn`'s.

In [ ]:
data = fetch_california_housing(as_frame=True).frame
income = data["MedInc"].to_numpy()
value  = data["MedHouseVal"].to_numpy()
print(len(income), "districts ready.")

## Step 2 — put income on a friendly scale

Income runs from about 0.5 to 15, value from 0 to 5. Those different scales make the downhill walk lurch about. One plain fix: **standardise** the income — subtract its average and divide by its spread — so it is centred on 0 with a spread of 1. The value we leave alone. (This scaling trick gets its own session, S12.)

In [ ]:
income_scaled = (income - income.mean()) / income.std()
print("scaled income runs from", round(income_scaled.min(), 2),
      "to", round(income_scaled.max(), 2))

## Step 3 — the score we push down (MSE)

Our line is `predicted = slope * income_scaled + start`. Learning means turning the two knobs — `slope` and `start` — until the model's score is as small as it goes.

The score here is the **MSE**: the average *squared* miss. Squaring keeps every miss positive (so pluses and minuses do not cancel) and punishes big misses harder. Let us write it as a plain function and check a deliberately bad guess.

In [ ]:
def predict(slope, start):
    return slope * income_scaled + start

def mse(slope, start):
    misses = value - predict(slope, start)
    return (misses ** 2).mean()

print("score of a bad guess (slope=0, start=0):", round(mse(0.0, 0.0), 3))

## Step 4 — which way is downhill?

From any guess we want to step in the direction that lowers the score fastest. Calculus gives that direction (the **gradient**); the computer works it out so you never differentiate by hand. For the mean squared miss it comes out as:

- push on **slope**: `2 * mean((prediction - value) * income_scaled)`
- push on **start**: `2 * mean(prediction - value)`

We move a small step *against* the gradient (downhill). How big that step is — the **learning rate** — is the dial we choose.

In [ ]:
def gradient(slope, start):
    error = predict(slope, start) - value       # signed miss per district
    grad_slope = 2 * np.mean(error * income_scaled)
    grad_start = 2 * np.mean(error)
    return grad_slope, grad_start

# One step from the bad guess, to see the score drop.
slope, start = 0.0, 0.0
learning_rate = 0.1
gs, gt = gradient(slope, start)
slope, start = slope - learning_rate * gs, start - learning_rate * gt
print("after ONE step -> score", round(mse(slope, start), 3))

## Step 5 — take many steps, and record the score each time

Now just repeat that one step a few hundred times, saving the score as we go so we can watch it fall.

In [ ]:
slope, start = 0.0, 0.0        # start over from the bad guess
learning_rate = 0.1
history = []

for step in range(300):
    gs, gt = gradient(slope, start)
    slope = slope - learning_rate * gs
    start = start - learning_rate * gt
    history.append(mse(slope, start))

print("final score :", round(history[-1], 4))
print("final slope :", round(slope, 3), " (in scaled units)")
print("final start :", round(start, 3))

## Step 6 — watch it learn

The score should drop steeply at first, then flatten as the line settles into the bottom of the bowl. That flattening curve *is* the model learning.

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(history, color="#2E75B6", linewidth=2)
plt.xlabel("step number")
plt.ylabel("score (mean squared miss)")
plt.title("The model learning: the score walks downhill")
plt.show()

## Step 7 — adjust the learning rate

One number controls the whole walk: the **learning rate**. Too small and the model crawls (300 steps aren't enough); too big and it overshoots and blows up; just right and it settles quickly. Run the same walk at several learning rates and compare the score after a fixed 60 steps.

In [ ]:
for learning_rate in [0.001, 0.01, 0.3, 1.2]:
    s, b = 0.0, 0.0
    for _ in range(60):
        gs, gt = gradient(s, b)
        s, b = s - learning_rate * gs, b - learning_rate * gt
    score = mse(s, b)
    tag = "diverged!" if not np.isfinite(score) or score > 1e3 else "ok"
    print(f"learning rate {learning_rate:<6} -> score after 60 steps: {score:>14.3f}   {tag}")

## Step 8 — did we reach the same line as scikit-learn?

The honest test of our hand-cranked walk: does it land where the library's one-shot answer lands? We fit `scikit-learn` on the same scaled income and compare the predictions. They should agree to a whisker.

In [ ]:
check = LinearRegression().fit(income_scaled.reshape(-1, 1), value)

our_line = predict(slope, start)
sklearn_line = check.predict(income_scaled.reshape(-1, 1))
biggest_gap = np.abs(our_line - sklearn_line).max()

print("sklearn slope/start:", round(check.coef_[0], 3), round(check.intercept_, 3))
print("our     slope/start:", round(slope, 3), round(start, 3))
print("biggest difference in predicted value:", round(biggest_gap, 5))

## Your turn (5-10 minutes)

1. In Step 5, change `range(300)` to `range(30)`. Did the model finish learning, or stop early? Look at the final score.
2. In Step 5, set `learning_rate` to `0.01`. Does the curve still flatten within 300 steps, or is it still crawling?
3. In a text cell, in your own words: what is the model *doing* on each step, and how does it know which way is downhill?

In [ ]:
# Your turn -- edit the loop above, or experiment here.


## What you just did

You built gradient descent from scratch — a score, its downhill direction, and a small step repeated — and watched it discover the same line `scikit-learn` finds instantly. "The model learns" is no longer mysterious: it is a ball rolling to the bottom of a bowl, one step at a time, with the **learning rate** setting how bold each step is. Every bigger model in this course, all the way to neural networks, learns by this same walk.

That completes S07. Next session (S08) tames the *too-complex* model and carries the housing project to a finished, honest result.